# 04 · Compare Them Head-to-Head  *(Students · CPU ok)*

This is the payoff. We send the **same questions** to four setups and read the answers side by side:

1. **Base** — the instruct model with no help
2. **From scratch** — the tiny model from Notebook 01
3. **Fine-tuned** — base + the LoRA adapter from Notebook 02
4. **RAG** — base + retrieval from Notebook 03

> **Before you run:** make sure you have the `artifacts/` folder the instructor shared (it holds the
> trained model, the LoRA adapter, the RAG index, the corpus, and the test prompts). This runs on CPU.

In [ ]:
import json, numpy as np, torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from sentence_transformers import SentenceTransformer
import faiss

ARTIFACTS = Path("artifacts")
BASE_MODEL = "Qwen/Qwen3-1.7B"
device = "cuda" if torch.cuda.is_available() else "cpu"
prompts = json.load(open(ARTIFACTS / "test_prompts.json"))
print("Device:", device, "| test prompts:", len(prompts))

### Load everything (one base model, shared to save memory)
We load the instruct model once and attach the LoRA adapter to it; turning the adapter on/off gives us
both the **base** and the **fine-tuned** answers from a single copy in memory.

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype="auto").to(device)
tuned = PeftModel.from_pretrained(base, str(ARTIFACTS / "lora_adapter"))  # adapter on top of base

# From-scratch tiny model
scratch_tok = AutoTokenizer.from_pretrained(ARTIFACTS / "scratch_model")
scratch = AutoModelForCausalLM.from_pretrained(ARTIFACTS / "scratch_model").to(device)

# RAG index
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
index = faiss.read_index(str(ARTIFACTS / "rag.faiss"))
docs = list(np.load(ARTIFACTS / "rag_docs.npy", allow_pickle=True))
print("All four setups loaded.")

In [ ]:
def chat(model, question, max_new_tokens=160):
    msgs = [{"role": "user", "content": question + " /no_think"}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                     enable_thinking=False)
    ids = tok(prompt, return_tensors="pt").to(device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def answer_base(q):
    with tuned.disable_adapter():     # adapter off => plain base model
        return chat(tuned, q)

def answer_tuned(q):
    return chat(tuned, q)             # adapter on => fine-tuned

def answer_scratch(q, n=40):
    ids = scratch_tok(q, return_tensors="pt").to(device)
    out = scratch.generate(**ids, max_new_tokens=n, do_sample=True, top_k=40,
                           pad_token_id=scratch_tok.eos_token_id)
    return scratch_tok.decode(out[0], skip_special_tokens=True)

def answer_rag(q, k=3):
    qv = embedder.encode([q], normalize_embeddings=True).astype("float32")
    ctx = "\n".join(f"- {docs[i]}" for i in index.search(qv, k)[1][0])
    with tuned.disable_adapter():     # RAG on the plain base model (no fine-tuning)
        return chat(tuned, f"Use ONLY this context; if it's not there, say so.\n"
                           f"Context:\n{ctx}\n\nQuestion: {q}")
print("Answer functions ready.")

### Run the comparison
For each test question, see all four answers and the note on **who should win**.

In [ ]:
for p in prompts:
    print("=" * 100)
    print(f"Q [{p['type']}]: {p['q']}")
    print(f"(expected to favor: {p['win']})\n")
    print("BASE        :", answer_base(p["q"])[:300]); print()
    print("FINE-TUNED  :", answer_tuned(p["q"])[:300]); print()
    print("RAG         :", answer_rag(p["q"])[:300]); print()
    print("FROM-SCRATCH:", answer_scratch(p["q"])[:200]); print()

## What you should notice
| Question type | Winner | Why |
|---|---|---|
| Private "Redlake" facts | **RAG** | Only RAG can see facts no model was trained on |
| Common definitions | Base / tuned / RAG | All know common knowledge |
| Specific format/style | **Fine-tuned** | It learned the house style |
| From-scratch, anything | (loses) | Too little data/compute — shows why we rarely train from scratch |

### The decision rule to remember
- **New or changing facts?** → **RAG**
- **New behavior, tone, or format?** → **Fine-tune**
- **A brand-new capability from nothing?** → **Train** (expensive — usually not worth it)

Same base model, same knowledge, three very different tools. Pick the one that matches your problem.